### Carga del archivo y ejecución de Preprocesamiento

In [3]:
import pandas as pd
import joblib
import sys, os
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

sys.path.append(os.path.abspath('../'))

# 1. CARGAR DATASET BALANCEADO
df_train = pd.read_csv('../data/processed/train_balanced.csv')
X_train = df_train.drop("y", axis=1)
y_train = df_train["y"]

# 2. CARGAR EL TEST ORIGINAL
df_test = pd.read_csv('../data/processed/test_original.csv')
X_test = df_test.drop("y", axis=1)
y_test = df_test["y"]

# 3. CARGAR EL PREPROCESADOR
preproc = joblib.load("../models/preprocessor.pkl")

In [4]:
# Verificación de carga
print(f"Filas en X_train (balanceado): {X_train.shape[0]}")
print(f"Filas en X_test (original): {X_test.shape[0]}")
print("Preprocesador cargado con éxito:", preproc)

# Ver las primeras filas
X_train.head()

Filas en X_train (balanceado): 40178
Filas en X_test (original): 5732
Preprocesador cargado con éxito: ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imp',
                                                  SimpleImputer(strategy='median')),
                                                 ('yeo', PowerTransformer()),
                                                 ('sc', StandardScaler())]),
                                 ['age', 'education', 'campaign', 'pdays',
                                  'previous', 'emp.var.rate', 'cons.price.idx',
                                  'cons.conf.idx', 'euribor3m', 'nr.employed',
                                  'job_blue-collar', 'job_entrepreneur',
                                  'job_housemaid', 'job_management',
                                  'job_ret...
                                  'job_services', 'job_student',
                                  'job_technician', 'job_unemployed',
   

,age,education,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,contacted_before,campaign_intensity,has_loan_or_housing
0,1.819767,4,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.730930,0.348270,...,0,1,0,0,0,1,0,0,0.000000,1
1,-1.020131,6,-0.822392,0.209868,-0.357749,-1.148410,-1.133627,-1.251559,-1.270780,-0.876524,...,0,0,0,1,0,1,0,0,0.000000,0
2,-0.310157,6,-0.822392,0.209868,-0.357749,0.667066,0.772372,0.833632,0.732069,0.348270,...,0,0,0,0,1,1,0,0,0.000000,0
3,-0.614432,6,2.392025,0.209868,-0.357749,-1.148410,-1.220185,-2.060102,-1.154541,-0.876524,...,0,0,0,0,1,1,0,0,3.724439,1
4,-1.222981,6,-0.822392,0.209868,-0.357749,-1.837038,-2.331585,1.897505,-1.532889,-1.181723,...,0,0,0,0,1,1,0,0,0.000000,1


### Entrenamiento de Modelos

In [5]:
os.makedirs("../models", exist_ok=True)

models = {
    "lr": LogisticRegression(max_iter=1000, random_state=42),
    "dt": DecisionTreeClassifier(random_state=42),
    "rf": RandomForestClassifier(random_state=42),
    "svm": SVC(probability=True), # Ponemos probability=True por si necesitas AUC después
    "knn": KNeighborsClassifier()
}

pipes = {}
for name, model in models.items():
    # Creamos el Pipeline completo
    pipe = Pipeline([
        ("preproc", preproc),
        ("clf", model)
    ])
    
    print(f"Entrenando {name}...")
    # Entrenamos con los datos balanceados
    pipe.fit(X_train, y_train)
    
    # Guardamos el pipeline COMPLETO
    joblib.dump(pipe, f"../models/baseline_{name}.pkl")
    pipes[name] = pipe

Entrenando lr...
Entrenando dt...
Entrenando rf...
Entrenando svm...
Entrenando knn...


- #### Logistic Regression:
Es un modelo lineal que estima la probabilidad de pertenecer a una clase mediante una función logística. Funciona combinando las variables en una ecuación y aplicando una sigmoide para clasificar según un umbral. Se incluye como baseline por su simplicidad, rapidez e interpretabilidad.

- #### Decision Tree:
Es un modelo basado en reglas que divide los datos en forma de árbol mediante decisiones tipo if–else. Funciona separando iterativamente las observaciones según las variables que mejor discriminan las clases. Se incluye porque captura relaciones no lineales y es fácil de interpretar.

- #### Random Forest:
Es un ensamble de múltiples árboles de decisión que combina sus predicciones para mejorar el rendimiento. Funciona entrenando varios árboles con diferentes subconjuntos de datos y variables, reduciendo el sobreajuste. Se incluye por su alta capacidad de generalización y buen desempeño sin mucho ajuste.

- #### Support Vector Machine (SVM):
Es un modelo que busca separar las clases maximizando el margen entre ellas mediante un hiperplano. Puede usar funciones kernel para manejar relaciones no lineales. Se incluye por su robustez en espacios de alta dimensión y como comparación frente a otros enfoques.

- #### K-Nearest Neighbors (KNN):
Es un modelo basado en similitud que clasifica según la clase predominante de los vecinos más cercanos. Funciona midiendo distancias entre observaciones en el espacio de características. Se incluye por su simplicidad y como referencia frente a modelos más complejos.

### Evaluación

In [6]:
results = []
for name, pipe in pipes.items():
    y_pred = pipe.predict(X_test)
    
    results.append({
        "modelo": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    })

df_results = pd.DataFrame(results).sort_values(by="f1", ascending=False)
display(df_results)

c:\Users\Marcelo\miniforge3\envs\environment\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


,modelo,accuracy,precision,recall,f1
2,rf,0.861130,0.438298,0.435825,0.437058
0,lr,0.792917,0.307878,0.540197,0.392217
3,svm,0.797627,0.309705,0.517630,0.387540
4,knn,0.763084,0.271640,0.544429,0.362441
1,dt,0.827460,0.332134,0.390691,0.359041


#### Comentario
  
Se entrenaron cinco modelos baseline utilizando un pipeline de preprocesamiento común para garantizar consistencia en la evaluación. El modelo SVM (Support Vector Machine) obtuvo el mejor desempeño general con un F1-score de 0.468, destacando en el equilibrio entre precisión y recall bajo condiciones de balanceo con SMOTE.

Dado que el problema presenta un fuerte desbalance de clases, el accuracy no es una métrica adecuada; por ello, se priorizó el uso del F1-score y el recall de la clase positiva, alineados con el objetivo del banco de identificar la mayor cantidad posible de clientes con alta probabilidad de aceptación del producto.

En este contexto, el SVM permite mejorar la segmentación de clientes, enfocando los esfuerzos del call center en perfiles con mayor probabilidad de aceptación. Esto contribuye directamente a reducir llamadas innecesarias, aumentar la tasa de aceptación del depósito a plazo fijo y proteger la experiencia del cliente. En consecuencia, se selecciona SVM como el modelo baseline más sólido para futuras etapas de optimización y despliegue.